<a href="https://colab.research.google.com/github/faizanarif2/worker-safety-helmet-detection/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Worker Safety Helmet Detection using YOLOv8

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os

project_path = "/content/drive/MyDrive/helmet_detection_project"

folders = [
    "dataset",
    "training_results",
    "models",
    "predictions"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created.")

Project folders created.


In [4]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 9.4 MB/s eta 0:00:00


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
GPU available: True


In [6]:
!nvidia-smi

Thu Sep 24 18:11:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from ultralytics import YOLO

print("Ultralytics YOLO imported successfully.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics YOLO imported successfully.


In [8]:
import os

dataset_folder = os.path.join(project_path, "dataset")

print("Dataset folder:", dataset_folder)
print("Files:", os.listdir(dataset_folder))

Dataset folder: /content/drive/MyDrive/helmet_detection_project/dataset
Files: ['helmet_dataset_version1.zip']


In [9]:
import zipfile
from pathlib import Path

zip_files = list(Path(dataset_folder).glob("*.zip"))

print("ZIP files:", [file.name for file in zip_files])

assert len(zip_files) == 1, "Upload exactly one dataset ZIP to the dataset folder."

zip_path = zip_files[0]
extract_path = Path("/content/helmet_dataset")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted.")
print("Extracted files:", os.listdir(extract_path))

ZIP files: ['helmet_dataset_version1.zip']
Dataset extracted.
Extracted files: ['README.roboflow.txt', 'test', 'valid', 'README.dataset.txt', 'train', 'data.yaml']


In [10]:
import yaml

yaml_files = list(extract_path.rglob("data.yaml"))

assert len(yaml_files) == 1, "Could not identify exactly one data.yaml file."

data_yaml = yaml_files[0]
dataset_root = data_yaml.parent

with open(data_yaml, "r") as file:
    data = yaml.safe_load(file)

data["path"] = str(dataset_root)
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

with open(data_yaml, "w") as file:
    yaml.safe_dump(data, file, sort_keys=False)

print("Dataset YAML:", data_yaml)
print("Class mapping:", data["names"])

names = data["names"]
class_names = set(names.values()) if isinstance(names, dict) else set(names)

assert class_names == {"person", "helmet", "no_helmet"}

print("All three classes verified.")

Dataset YAML: /content/helmet_dataset/data.yaml
Class mapping: ['helmet', 'no_helmet', 'person']
All three classes verified.


In [11]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

for split in ["train", "valid", "test"]:
    image_folder = dataset_root / split / "images"
    label_folder = dataset_root / split / "labels"

    images = [
        file for file in image_folder.iterdir()
        if file.suffix.lower() in image_extensions
    ]

    labels = list(label_folder.glob("*.txt"))

    print(f"{split}: {len(images)} images, {len(labels)} label files")

train: 221 images, 221 label files
valid: 63 images, 63 label files
test: 32 images, 32 label files
